In [ ]:
# Uncomment the lines below to create a ZIP and trigger browser download

# import zipfile
# from google.colab import files
#
# zip_path = "/content/geographic_data.zip"
# with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for fp in sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.json"))):
#         zf.write(fp, os.path.basename(fp))
#         print(f"  Added: {os.path.basename(fp)}")
#
# size_mb = os.path.getsize(zip_path) / (1024 * 1024)
# print(f"\nZIP created: {size_mb:.1f} MB")
# files.download(zip_path)

print("Uncomment this cell to download files as a ZIP to your computer.")

## Step 6: Download as ZIP (alternative to Drive)

If you prefer to download the files directly to your computer instead of Google Drive.

In [ ]:
import shutil

if not DRIVE_FOLDER:
    print("Google Drive not enabled — skipping.")
    print("Files are in: /content/geographic_data/")
    print("To download manually: click the folder icon (left sidebar) → navigate → right-click → Download")
else:
    print(f"Copying files to Google Drive: {DRIVE_FOLDER}\n")
    json_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.json")))
    for fp in json_files:
        dest = os.path.join(DRIVE_FOLDER, os.path.basename(fp))
        shutil.copy2(fp, dest)
        size_mb = os.path.getsize(dest) / (1024 * 1024)
        print(f"  ✓ {os.path.basename(fp)} ({size_mb:.1f} MB)")

    print(f"\n✓ {len(json_files)} files copied to Google Drive!")
    print(f"  Location: {DRIVE_FOLDER}")
    print("  These files will persist after the Colab session ends.")

## Step 5: Copy to Google Drive

Copies the downloaded JSON files to Google Drive so they persist after the Colab session ends.

In [ ]:
import glob

print("═══ Downloaded Files Summary ═══\n")
json_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.json")))

if not json_files:
    print("No JSON files found! Run the download step first.")
else:
    total_records = 0
    for fp in json_files:
        with open(fp) as f:
            data = json.load(f)
        meta = data.get("_metadata", {})
        count = meta.get("recordCount", 0)
        total_records += count
        size_mb = os.path.getsize(fp) / (1024 * 1024)
        print(f"  {os.path.basename(fp)}")
        print(f"    Records: {count:,}  |  Size: {size_mb:.1f} MB  |  Downloaded: {meta.get('downloadedAt','?')[:19]}")

        # Show sample record
        records = data.get("data", [])
        if records:
            sample = records[0]
            name = sample.get("name") or sample.get("fips") or "?"
            centroid = sample.get("centroid")
            print(f"    Sample: {name} (centroid: {centroid})")
        print()

    print(f"  Total: {total_records:,} records across {len(json_files)} files")

    if total_records == 0:
        print("\n  ⚠ WARNING: All files have 0 records!")
        print("  The Census TIGERweb API may be down or changed.")
        print("  Check: https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb")

## Step 4: Preview Downloaded Data

Quick sanity check — shows record counts and sample data for each file.

In [ ]:
import time as _t
_start = _t.time()

state_arg = STATE_FIPS.strip() if STATE_FIPS.strip() else None
scope = DOWNLOAD_SCOPE
saved_files = []
counties_data = None

print("=" * 60)
print("  CRASH LENS — Geographic Data Downloader (Colab)")
print(f"  Date: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")
if state_arg:
    print(f"  Scope: State FIPS {state_arg}")
else:
    print(f"  Scope: {'Nationwide (50 states + DC)' if scope == 'all' else scope}")
print("=" * 60)

# Discover working TIGERweb service
discover_tiger_service()

# ── States ──
if scope in ("all", "states") and not state_arg:
    states = download_states()
    saved_files.append(save_json(states, "us_states.json"))

# ── Counties ──
if scope in ("all", "counties", "counties-and-mpos"):
    counties_data = download_counties(state_fips=state_arg)
    suffix = f"_{state_arg}" if state_arg else ""
    saved_files.append(save_json(counties_data, f"us_counties{suffix}.json"))

# ── Incorporated Places ──
if scope in ("all", "places"):
    places = download_places(state_fips=state_arg)
    suffix = f"_{state_arg}" if state_arg else ""
    saved_files.append(save_json(places, f"us_places{suffix}.json"))

# ── County Subdivisions ──
if scope in ("all", "subdivisions"):
    if SKIP_SUBDIVISIONS and scope == "all":
        print("\n⏭ Skipping county subdivisions (SKIP_SUBDIVISIONS=True)")
    else:
        subdivisions = download_county_subdivisions(state_fips=state_arg)
        suffix = f"_{state_arg}" if state_arg else ""
        saved_files.append(save_json(subdivisions, f"us_county_subdivisions{suffix}.json"))

# ── MPOs ──
if scope in ("all", "mpos", "counties-and-mpos"):
    mpos = download_mpos()
    if counties_data:
        mpos = assign_counties_to_mpos(mpos, counties_data)
    elif scope == "mpos":
        print("\n  (Downloading counties for MPO-county assignment...)")
        counties_data = download_counties(state_fips=state_arg)
        mpos = assign_counties_to_mpos(mpos, counties_data)
    saved_files.append(save_json(mpos, "us_mpos.json"))

elapsed = _t.time() - _start
print("\n" + "=" * 60)
print(f"  ✓ Download complete! ({elapsed/60:.1f} minutes)")
print(f"  Files saved: {len(saved_files)}")
for fp in saved_files:
    size = os.path.getsize(fp) / (1024*1024)
    print(f"    {os.path.basename(fp)}: {size:.1f} MB")
print("=" * 60)

## Step 3: Run the Download

This cell discovers the active Census TIGERweb endpoint, downloads the selected layers, and saves JSON files. Full nationwide download takes ~30-90 minutes depending on Colab's network speed.

In [ ]:
# ─── Save helper ────────────────────────────────────────────────────────────

def save_json(data, filename):
    """Save data to JSON file with metadata header."""
    filepath = os.path.join(OUTPUT_DIR, filename)
    output = {
        "_metadata": {
            "source": "Census TIGERweb + BTS/USDOT NTAD",
            "downloadedAt": datetime.now(timezone.utc).isoformat(),
            "script": "CRASH_LENS_Geographic_Data_Download.ipynb (Google Colab)",
            "recordCount": len(data),
        },
        "data": data,
    }
    with open(filepath, "w") as f:
        json.dump(output, f, indent=2)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"  Saved {filename} ({size_mb:.1f} MB, {len(data)} records)")
    return filepath

print("Save function loaded.")

In [ ]:
# ─── Layer download functions ────────────────────────────────────────────────

def download_states():
    """Download all US states + DC from TIGERweb."""
    print("\n═══ Downloading States ═══")
    url = resolve_layer_url("states")
    features = arcgis_query_all(
        url, where="1=1",
        out_fields="GEOID,NAME,STUSAB,FUNCSTAT,ALAND,AWATER",
        return_geometry=True, label="states"
    )
    states = []
    for f in features:
        attr = f.get("attributes", {})
        geoid = attr.get("GEOID", "")
        if geoid not in VALID_STATE_FIPS:
            continue
        centroid, bbox = compute_centroid_from_rings(f.get("geometry", {}))
        states.append({
            "fips": geoid, "name": attr.get("NAME", ""),
            "abbreviation": attr.get("STUSAB", ""),
            "centroid": centroid, "bbox": bbox,
            "landAreaSqM": attr.get("ALAND"),
            "waterAreaSqM": attr.get("AWATER"),
        })
    states.sort(key=lambda s: s["fips"])
    print(f"  ✓ {len(states)} states processed")
    return states


def download_counties(state_fips=None):
    """Download all US counties from TIGERweb."""
    print("\n═══ Downloading Counties ═══")
    url = resolve_layer_url("counties")
    where = f"STATE='{state_fips}'" if state_fips else "1=1"
    features = arcgis_query_all(
        url, where=where,
        out_fields="GEOID,STATE,COUNTY,NAME,LSAD,FUNCSTAT,ALAND,AWATER",
        return_geometry=True, label="counties"
    )
    counties = []
    for f in features:
        attr = f.get("attributes", {})
        state = attr.get("STATE", "")
        if state not in VALID_STATE_FIPS:
            continue
        centroid, bbox = compute_centroid_from_rings(f.get("geometry", {}))
        counties.append({
            "stateFips": state, "countyFips": attr.get("COUNTY", ""),
            "geoid": attr.get("GEOID", ""), "name": attr.get("NAME", ""),
            "lsad": attr.get("LSAD", ""),
            "centroid": centroid, "bbox": bbox,
            "landAreaSqM": attr.get("ALAND"),
        })
    counties.sort(key=lambda c: c["geoid"])
    print(f"  ✓ {len(counties)} counties processed")
    return counties


def download_places(state_fips=None):
    """Download all incorporated places from TIGERweb."""
    print("\n═══ Downloading Incorporated Places ═══")
    url = resolve_layer_url("places")
    where = f"STATE='{state_fips}'" if state_fips else "1=1"
    features = arcgis_query_all(
        url, where=where,
        out_fields="GEOID,STATE,PLACEFP,NAME,NAMELSAD,LSAD,FUNCSTAT,ALAND",
        return_geometry=True, label="places"
    )
    places = []
    for f in features:
        attr = f.get("attributes", {})
        state = attr.get("STATE", "")
        if state not in VALID_STATE_FIPS:
            continue
        funcstat = attr.get("FUNCSTAT", "")
        if funcstat not in ("A", "S"):
            continue
        centroid, bbox = compute_centroid_from_rings(f.get("geometry", {}))
        lsad = attr.get("LSAD", "")
        places.append({
            "stateFips": state, "placeFips": attr.get("PLACEFP", ""),
            "geoid": attr.get("GEOID", ""), "name": attr.get("NAME", ""),
            "fullName": attr.get("NAMELSAD", ""), "lsad": lsad,
            "type": LSAD_PLACE_TYPE.get(lsad, "other"),
            "funcstat": funcstat, "centroid": centroid, "bbox": bbox,
            "landAreaSqM": attr.get("ALAND"),
        })
    places.sort(key=lambda p: p["geoid"])
    print(f"  ✓ {len(places)} places processed")
    return places


def download_county_subdivisions(state_fips=None):
    """Download county subdivisions (MCDs, townships, towns) from TIGERweb."""
    print("\n═══ Downloading County Subdivisions ═══")
    url = resolve_layer_url("county_subdivisions")
    where = f"STATE='{state_fips}'" if state_fips else "1=1"
    features = arcgis_query_all(
        url, where=where,
        out_fields="GEOID,STATE,COUNTY,COUSUBFP,NAME,NAMELSAD,LSAD,FUNCSTAT",
        return_geometry=True, label="county subdivisions"
    )
    subdivisions = []
    for f in features:
        attr = f.get("attributes", {})
        state = attr.get("STATE", "")
        if state not in VALID_STATE_FIPS:
            continue
        funcstat = attr.get("FUNCSTAT", "")
        if funcstat not in ("A", "S"):
            continue
        centroid, bbox = compute_centroid_from_rings(f.get("geometry", {}))
        subdivisions.append({
            "stateFips": state, "countyFips": attr.get("COUNTY", ""),
            "cousubFips": attr.get("COUSUBFP", ""),
            "geoid": attr.get("GEOID", ""), "name": attr.get("NAME", ""),
            "fullName": attr.get("NAMELSAD", ""),
            "lsad": attr.get("LSAD", ""), "funcstat": funcstat,
            "centroid": centroid, "bbox": bbox,
        })
    subdivisions.sort(key=lambda s: s["geoid"])
    print(f"  ✓ {len(subdivisions)} county subdivisions processed")
    return subdivisions


def download_mpos():
    """Download MPO boundaries from BTS/USDOT NTAD service."""
    print("\n═══ Downloading MPOs (BTS/USDOT) ═══")
    features = []
    for mpo_url in BTS_MPO_URLS:
        short_name = mpo_url.split("/services/")[1].split("/query")[0]
        print(f"  Trying: {short_name}...")
        features = arcgis_query_all(
            mpo_url, where="1=1", out_fields="*",
            return_geometry=True, label="MPOs"
        )
        if features:
            break

    mpos = []
    for f in features:
        attr = f.get("attributes", {})
        geometry = f.get("geometry", {})
        centroid, bbox = compute_centroid_from_rings(geometry)
        mpos.append({
            "name": attr.get("MPO_NAME") or attr.get("NAME") or attr.get("MpoName") or "",
            "acronym": attr.get("MPO_ACRONYM") or attr.get("ACRONYM") or attr.get("MpoAcronym") or "",
            "mpoId": attr.get("MPO_ID") or attr.get("OBJECTID") or attr.get("MpoId") or "",
            "stateFips": attr.get("STATE_FIPS") or attr.get("STFIPS") or attr.get("StateFips") or "",
            "stateAbbr": attr.get("STATE") or attr.get("STUSAB") or attr.get("StateAbbr") or "",
            "centroid": centroid, "bbox": bbox,
            "population": attr.get("POP") or attr.get("POPULATION") or attr.get("Pop") or None,
            "status": attr.get("STATUS") or attr.get("Status") or "",
            "_rawFields": {k: v for k, v in attr.items() if v is not None and v != ""},
        })
    mpos.sort(key=lambda m: (m.get("stateAbbr", ""), m.get("name", "")))
    print(f"  ✓ {len(mpos)} MPOs processed")
    return mpos


def assign_counties_to_mpos(mpos, counties):
    """Assign counties to MPOs using centroid-in-bbox approximation."""
    print("\n═══ Assigning Counties to MPOs (centroid-in-bbox) ═══")
    for mpo in mpos:
        mpo["counties"] = []
        mpo_bbox = mpo.get("bbox")
        if not mpo_bbox:
            continue
        west, south, east, north = mpo_bbox
        for county in counties:
            c = county.get("centroid")
            if not c:
                continue
            lon, lat = c[0], c[1]
            if west <= lon <= east and south <= lat <= north:
                mpo["counties"].append({
                    "stateFips": county["stateFips"],
                    "countyFips": county["countyFips"],
                    "geoid": county["geoid"],
                    "name": county["name"],
                })
    assigned = sum(1 for m in mpos if m.get("counties"))
    print(f"  ✓ {assigned}/{len(mpos)} MPOs have county assignments")
    return mpos


print("Layer download functions loaded.")

In [ ]:
# ─── Dynamic TIGERweb service discovery ─────────────────────────────────────

def discover_tiger_service():
    """Try each TIGERweb service URL until one responds with valid layers."""
    global _tiger_base, _discovered_layers

    print("═══ Discovering Census TIGERweb Service ═══")
    for service_url in TIGER_SERVICES:
        service_name = service_url.split("/services/")[1].split("/MapServer")[0]
        try:
            resp = requests.get(service_url, params={"f": "json"}, timeout=30)
            resp.raise_for_status()
            data = resp.json()

            if "error" in data:
                print(f"  ⚠ {service_name}: API error — {data['error'].get('message', '?')}")
                continue

            layers = data.get("layers", [])
            if not layers:
                print(f"  ⚠ {service_name}: No layers found")
                continue

            layer_lookup = {layer["name"]: layer["id"] for layer in layers}
            resolved = {}
            for key, candidate_names in TIGER_LAYER_NAMES.items():
                for name in candidate_names:
                    if name in layer_lookup:
                        resolved[key] = layer_lookup[name]
                        break

            if len(resolved) >= 2:
                _tiger_base = service_url
                _discovered_layers = resolved
                print(f"  ✓ Using {service_name}")
                for key, layer_id in resolved.items():
                    print(f"    {key}: layer {layer_id}")
                return
            else:
                print(f"  ⚠ {service_name}: Only matched {len(resolved)}/{len(TIGER_LAYER_NAMES)} layers")

        except requests.exceptions.RequestException as e:
            print(f"  ⚠ {service_name}: {e}")

    print("  ⚠ Dynamic discovery failed — will try fallback layer IDs")
    _tiger_base = TIGER_SERVICES[0]
    _discovered_layers = None


def resolve_layer_url(layer_key):
    """Get the query URL for a layer, trying discovered IDs then fallbacks."""
    if _discovered_layers and layer_key in _discovered_layers:
        return f"{_tiger_base}/{_discovered_layers[layer_key]}/query"

    for layer_id in TIGER_LAYERS_FALLBACK.get(layer_key, []):
        url = f"{_tiger_base}/{layer_id}/query"
        try:
            resp = requests.get(url, params={
                "where": "1=1", "outFields": "GEOID",
                "returnGeometry": "false", "f": "json",
                "resultRecordCount": 1
            }, timeout=15)
            data = resp.json()
            if data.get("features"):
                print(f"  ✓ Found {layer_key} at layer {layer_id}")
                return url
        except Exception:
            continue

    fallback_id = TIGER_LAYERS_FALLBACK.get(layer_key, [0])[0]
    return f"{_tiger_base}/{fallback_id}/query"


print("Service discovery functions loaded.")

In [ ]:
# ─── ArcGIS paginated query ─────────────────────────────────────────────────

def arcgis_query_all(url, where="1=1", out_fields="*", return_geometry=True,
                     out_sr=4326, page_size=PAGE_SIZE, label="records"):
    """Paginate through an ArcGIS REST API query endpoint with error handling."""
    all_features = []
    offset = 0

    while True:
        params = {
            "where": where,
            "outFields": out_fields,
            "returnGeometry": "true" if return_geometry else "false",
            "outSR": out_sr,
            "f": "json",
            "resultOffset": offset,
            "resultRecordCount": page_size,
        }

        data = None
        for attempt in range(MAX_RETRIES):
            try:
                resp = requests.get(url, params=params, timeout=60)
                resp.raise_for_status()
                data = resp.json()
                break
            except Exception as e:
                wait = 2 ** (attempt + 1)
                print(f"  ⚠ Request failed (attempt {attempt+1}/{MAX_RETRIES}): {e}")
                if attempt < MAX_RETRIES - 1:
                    print(f"    Retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    print(f"  ✗ Giving up after {MAX_RETRIES} attempts")
                    return all_features

        if data is None:
            break

        # ArcGIS returns errors with HTTP 200 — must check JSON body
        if "error" in data:
            err = data["error"]
            print(f"  ✗ ArcGIS API error (code {err.get('code','?')}): {err.get('message','Unknown')}")
            for d in err.get("details", [])[:3]:
                print(f"    → {d}")
            return all_features

        features = data.get("features", [])
        if not features:
            if offset == 0:
                print(f"  ⚠ API returned 0 features. Response keys: {list(data.keys())}")
                if len(str(data)) < 500:
                    print(f"    Response: {json.dumps(data, indent=None)[:400]}")
            break

        all_features.extend(features)
        print(f"  Fetched {len(all_features)} {label}...   ", end="\r")

        exceeded = data.get("exceededTransferLimit", False)
        if len(features) < page_size and not exceeded:
            break

        offset += len(features)
        time.sleep(REQUEST_DELAY)

    print(f"  Fetched {len(all_features)} {label} total.     ")
    return all_features


def compute_centroid_from_rings(geometry):
    """Compute centroid and bbox from polygon rings."""
    rings = geometry.get("rings", [])
    if not rings:
        return None, None
    all_x, all_y = [], []
    for ring in rings:
        for coord in ring:
            all_x.append(coord[0])
            all_y.append(coord[1])
    if not all_x:
        return None, None
    centroid = [round(sum(all_x)/len(all_x), 6), round(sum(all_y)/len(all_y), 6)]
    bbox = [round(min(all_x), 6), round(min(all_y), 6),
            round(max(all_x), 6), round(max(all_y), 6)]
    return centroid, bbox


print("Query and geometry functions loaded.")

In [ ]:
import json
import time
import requests
from pathlib import Path
from datetime import datetime, timezone

# ─── Census TIGERweb endpoints (ordered by preference) ──────────────────────
# The Census Bureau periodically retires vintage services, so we try multiple.
TIGER_SERVICES = [
    "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/tigerWMS_Current/MapServer",
    "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/tigerWMS_Census2020/MapServer",
    "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/tigerWMS_ACS2024/MapServer",
    "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/tigerWMS_ACS2023/MapServer",
]

# Layer name -> expected ArcGIS layer names (for dynamic lookup)
TIGER_LAYER_NAMES = {
    "states": ["States", "Census States"],
    "counties": ["Counties", "Census Counties"],
    "places": ["Places", "Incorporated Places", "Census Places"],
    "county_subdivisions": ["County Subdivisions", "Census County Subdivisions"],
}

# Fallback layer IDs (last known good)
TIGER_LAYERS_FALLBACK = {
    "states": [80, 54, 82, 84, 86, 88, 90],
    "counties": [82, 86, 100, 102],
    "places": [28, 30, 150, 152],
    "county_subdivisions": [30, 32, 160, 162],
}

# BTS/USDOT MPO endpoints
BTS_MPO_URLS = [
    "https://geo.dot.gov/server/rest/services/NTAD/Metropolitan_Planning_Organizations/MapServer/0/query",
    "https://geo.dot.gov/server/rest/services/NTAD/Metropolitan_Planning_Organizations/FeatureServer/0/query",
    "https://geo.dot.gov/server/rest/services/Hosted/Metropolitan_Planning_Organizations/FeatureServer/0/query",
    "https://services.arcgis.com/xOi1kZaI0eWDREZv/arcgis/rest/services/Metropolitan_Planning_Organizations/FeatureServer/0/query",
]

# 50 states + DC FIPS codes
VALID_STATE_FIPS = {
    "01","02","04","05","06","08","09","10","11","12",
    "13","15","16","17","18","19","20","21","22","23",
    "24","25","26","27","28","29","30","31","32","33",
    "34","35","36","37","38","39","40","41","42","44",
    "45","46","47","48","49","50","51","53","54","55","56"
}

# LSAD codes for places
LSAD_PLACE_TYPE = {
    "25": "city", "43": "town", "47": "village",
    "53": "CDP", "55": "CDP", "57": "CDP",
    "21": "borough", "44": "township", "46": "plantation",
    "49": "charter_township", "00": "city",
}

REQUEST_DELAY = 0.25
MAX_RETRIES = 4
PAGE_SIZE = 2000

# ─── Resolved at runtime ────────────────────────────────────────────────────
_tiger_base = None
_discovered_layers = None

print("Core constants loaded.")

## Step 2: Core Functions

ArcGIS REST API helpers, service discovery, and geometry utilities.

In [ ]:
DRIVE_FOLDER = None

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_FOLDER = '/content/drive/MyDrive/CRASH_LENS_Data/geographic'
    os.makedirs(DRIVE_FOLDER, exist_ok=True)
    print(f"Drive folder ready: {DRIVE_FOLDER}")
else:
    print("Skipping Google Drive — files will be in /content/geographic_data only")

## Step 1: Mount Google Drive (optional)

In [ ]:
# @title Download Settings { run: "auto" }

# @markdown **What to download:**
DOWNLOAD_SCOPE = "all"  # @param ["all", "states", "counties", "places", "subdivisions", "mpos", "counties-and-mpos"]

# @markdown **Limit to a single state?** Leave blank for nationwide.
STATE_FIPS = ""  # @param {type:"string"}

# @markdown **Skip county subdivisions?** (saves ~15 min on full download)
SKIP_SUBDIVISIONS = True  # @param {type:"boolean"}

# @markdown **Save to Google Drive?**
SAVE_TO_DRIVE = True  # @param {type:"boolean"}

# @markdown ---
# @markdown **Common State FIPS codes:** `04`=AZ, `06`=CA, `08`=CO, `10`=DE, `12`=FL, `13`=GA, `24`=MD, `34`=NJ, `36`=NY, `48`=TX, `51`=VA

import os
OUTPUT_DIR = "/content/geographic_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if STATE_FIPS:
    print(f"Scope: State FIPS {STATE_FIPS}")
else:
    print(f"Scope: {'Nationwide' if DOWNLOAD_SCOPE == 'all' else DOWNLOAD_SCOPE}")
print(f"Output: {OUTPUT_DIR}")
if SAVE_TO_DRIVE:
    print("Google Drive: Will save after download")

## Configuration

Choose what to download and where to save. Run this cell first, then run all remaining cells.

# CRASH LENS — Geographic Data Downloader

Downloads comprehensive US geographic/administrative boundary data from federal APIs and saves to Google Drive.

**Data Sources:**
| Layer | Source | Records |
|-------|--------|---------|
| States | Census TIGERweb | 51 (50 states + DC) |
| Counties | Census TIGERweb | ~3,200 |
| Incorporated Places | Census TIGERweb | ~30,000 cities/towns/villages/CDPs |
| County Subdivisions | Census TIGERweb | ~35,000 MCDs/townships |
| MPOs | BTS/USDOT NTAD | ~400 Metropolitan Planning Organizations |

**Output:** JSON files with FIPS codes, centroids, bounding boxes, and metadata.

---